# From statistical significance to decision value

## A reproducible comparison of PCA, regression and complementary evidence

The question is not *which single method is correct?* It is: **what different information do our methods recover, and does integrating that evidence improve an actual decision?**

We will separate four notions:

1. **Structure:** covariance spectrum, PCA and VIF.
2. **Inference:** conditional coefficient uncertainty and joint tests.
3. **Generalisation:** OLS, ridge, elastic net, PCR and PLS on independent observations.
4. **Decision:** expected payoff and regret after paying a known cost.

This notebook uses synthetic data with a known conditional mean. That mean is used only as an evaluation oracle and never enters fitting, diagnostics or model selection. The conclusions are conditional on the data-generating mechanisms examined here.

### Mathematical framework

For centred predictors `X`, PCA diagonalises their sample covariance:

$$S = \frac{X^{\top} X}{n-1}=V\Lambda V^{\top},$$

and constructs component scores `Z=XV`. Its first directions maximise **predictor variance**, not relevance to a target `Y`.

By contrast, a linear-regression p-value assesses an individual coefficient conditional on the remaining specified regressors and modelling assumptions. Under multicollinearity, large standard errors can make individual tests imprecise while a **joint F-test** remains informative.

Our simulated decision takes an action whenever a fitted model predicts that its benefit exceeds cost `c`:

$$a_{\widehat{m}}(x)=\mathbb{1}\{\widehat{m}(x)>c\}.$$

Writing `m(x)=E[Y\mid X=x]` for the *known simulation truth*, its expected net payoff is `E[(m(X)-c)a(X)]`. Regret is

$$R(\widehat{m})=E[(m(X)-c)_+]-E[(m(X)-c)a_{\widehat{m}}(X)].$$

This is expected opportunity loss, not classification accuracy or RMSE. In real interventions, `m(x)` cannot automatically be interpreted as a treatment effect; that requires a causal design.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

from analysis import run_experiment, simulate_data

RANDOM_SEED: int = 2026
COST: float = 0.6
BOOTSTRAPS: int = int(os.environ.get("EVIDENCE_BOOTSTRAPS", "150"))
REPEATS: int = int(os.environ.get("EVIDENCE_REPEATS", "12"))
if BOOTSTRAPS < 10 or REPEATS < 1:
    raise ValueError("Require at least 10 bootstrap draws and one Monte Carlo replicate")

## 1. Two correlated-predictor mechanisms

We construct eight latent factors, each represented by two highly correlated measurements. The resulting 16-dimensional predictor matrix is nearly eight-dimensional, but not exactly rank-deficient.

- **Low-variance signal:** `E[Y | X] = 10(X1-X2)`.
- **High-variance signal:** `E[Y | X] = 1.2(X1+X2)`.

In the first mechanism, PCA directions that explain almost no predictor variance may be the most useful for predicting the response. The second mechanism is a deliberately favourable case for variance-based reduction.

The training/test split is generated before fitting anything. Only the training set is inspected in the next three sections.

In [ ]:
low = simulate_data(seed=RANDOM_SEED, scenario="low_variance_signal")
high = simulate_data(seed=RANDOM_SEED, scenario="high_variance_signal")
features: list[str] = [f"x{i + 1:02d}" for i in range(low.x_train.shape[1])]
training = pd.DataFrame(low.x_train, columns=features)
design = sm.add_constant(training, has_constant="add")
ols_fit = sm.OLS(low.y_train, design).fit()
display(training.iloc[:5, :6])
print(f"Training observations: {len(training)}; predictors: {len(features)}")
print(f"Within-pair correlation: {training['x01'].corr(training['x02']):.3f}")

## 2. Structural evidence: PCA and collinearity

Standardise predictors using training observations only. Inspect the complete eigenvalue spectrum, including *small* directions, and compare explained variance with the **training-response** associations of the components.

The association diagnostic is descriptive; it cannot by itself establish predictive value outside the training sample. VIF and the condition number complement the PCA view: they describe different consequences of correlation and near-singularity.

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(low.x_train)
pca = PCA().fit(x_scaled)
scores = pca.transform(x_scaled)
signal_correlations = [
    float(np.corrcoef(scores[:, i], low.y_train)[0, 1])
    for i in range(scores.shape[1])
]
spectrum = pd.DataFrame(
    {
        "component": np.arange(1, len(features) + 1),
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained": np.cumsum(pca.explained_variance_ratio_),
        "squared_training_y_correlation": np.square(signal_correlations),
    }
)
vif = pd.DataFrame(
    {
        "feature": features,
        "VIF": [
            variance_inflation_factor(design.to_numpy(), j + 1)
            for j in range(len(features))
        ],
    }
)
print(f"Scaled-design condition number: {np.linalg.cond(x_scaled):.1f}")
print(f"Components retained at 95% X variance: {int(np.searchsorted(spectrum['cumulative_explained'], 0.95)) + 1}")
display(spectrum)
display(vif.sort_values("VIF", ascending=False).head(8))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(spectrum["component"], spectrum["cumulative_explained"], marker="o")
axes[0].axhline(0.95, color="grey", linestyle="--", label="95% X variance")
axes[0].set(xlabel="Principal component", ylabel="Cumulative X variance", title="PCA variance")
axes[0].legend()
axes[1].scatter(
    spectrum["explained_variance_ratio"],
    spectrum["squared_training_y_correlation"],
)
axes[1].set(
    xlabel="Fraction of X variance",
    ylabel="Squared training correlation with Y",
    title="Variance is not predictive relevance",
)
fig.tight_layout()
plt.show()

## 3. Regression inference and joint evidence

For the original predictors, fit a full OLS model and examine coefficient estimates, confidence intervals and p-values. Test the *pair* `(X1, X2)` jointly, conditional on all other columns.

A large individual p-value does not demonstrate irrelevance; a small p-value does not establish causality or practical importance. We therefore keep the inferential evidence visible instead of turning it into a single feature score.

In [ ]:
confidence = ols_fit.conf_int()
coefficient_summary = pd.DataFrame(
    {
        "term": ols_fit.params.index,
        "estimate": ols_fit.params.to_numpy(),
        "p_value": ols_fit.pvalues.to_numpy(),
        "ci_lower": confidence.iloc[:, 0].to_numpy(),
        "ci_upper": confidence.iloc[:, 1].to_numpy(),
    }
)
display(coefficient_summary.head(8))
joint_test = ols_fit.f_test("x01 = 0, x02 = 0")
print(f"Joint F test for X1 and X2: F={float(joint_test.fvalue):.3f}, p={float(joint_test.pvalue):.3g}")
print(f"Adjusted R² (training): {ols_fit.rsquared_adj:.3f}")

### Bootstrap coefficient stability

With correlated predictors, uncertainty in *individual coefficients* can remain large even if fitted predictions are useful. We bootstrap full-model OLS coefficients without making a preliminary variable-selection decision.

The empirical 95% intervals and sign frequencies are stability diagnostics. They are not substitutes for valid post-selection inference or a causal identification strategy.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED + 1)
n_observations, n_predictors = low.x_train.shape
coefficient_draws = np.empty((BOOTSTRAPS, n_predictors), dtype=np.float64)
for draw in range(BOOTSTRAPS):
    sample_index = rng.integers(0, n_observations, size=n_observations)
    boot_design = np.column_stack(
        [np.ones(n_observations), low.x_train[sample_index]]
    )
    # Least squares on the resample; the intercept is the first coefficient.
    coefficient_draws[draw] = np.linalg.lstsq(
        boot_design, low.y_train[sample_index], rcond=None
    )[0][1:]

lower, upper = np.quantile(coefficient_draws, [0.025, 0.975], axis=0)
bootstrap_summary = pd.DataFrame(
    {
        "feature": features,
        "bootstrap_lower_95": lower,
        "bootstrap_upper_95": upper,
        "fraction_positive": np.mean(coefficient_draws > 0, axis=0),
        "full_model_p": ols_fit.pvalues.iloc[1:].to_numpy(),
    }
)
display(bootstrap_summary.head(8))

## 4. All principal components versus truncation

If all components are retained, OLS after PCA is merely a rotation of the same full-rank predictor space. Fitted predictions must agree numerically with ordinary OLS.

The *dimension-reduction choice*—for example, retaining 95% predictor variance—is what can lose response information.

In [ ]:
raw_ols = make_pipeline(StandardScaler(), LinearRegression())
all_pc_ols = make_pipeline(
    StandardScaler(), PCA(n_components=low.x_train.shape[1]), LinearRegression()
)
pc95_ols = make_pipeline(
    StandardScaler(), PCA(n_components=0.95), LinearRegression()
)
for estimator in (raw_ols, all_pc_ols, pc95_ols):
    estimator.fit(low.x_train, low.y_train)
raw_prediction = np.asarray(raw_ols.predict(low.x_test)).reshape(-1)
all_pc_prediction = np.asarray(all_pc_ols.predict(low.x_test)).reshape(-1)
truncated_prediction = np.asarray(pc95_ols.predict(low.x_test)).reshape(-1)
assert np.allclose(raw_prediction, all_pc_prediction, rtol=1e-8, atol=1e-8)
print(f"Max full-PCA vs OLS prediction difference: {np.max(np.abs(raw_prediction - all_pc_prediction)):.2e}")
print(f"Test RMSE — OLS: {np.sqrt(np.mean((low.y_test - raw_prediction) ** 2)):.3f}")
print(f"Test RMSE — PCR 95%: {np.sqrt(np.mean((low.y_test - truncated_prediction) ** 2)):.3f}")

## 5. Independent test-set comparison

We now repeat both data-generating mechanisms. Candidates include full OLS, ridge, elastic net, PCR (95% of X variance), PLS (two components), and intentionally naive p-value screening.

A decision-guided option uses **only an internal split of the training set** to select a candidate according to observed average net benefit. It refits that candidate on all training observations and is finally assessed on a fresh external test set.

The comparison is deliberately not presented as a universal ranking. Observe both RMSE and regret; the best predictive model need not maximise expected decision value.

In [ ]:
experiment = run_experiment(
    repeats=REPEATS,
    n_train=180,
    n_test=600,
    cost=COST,
    seed=RANDOM_SEED,
)
summary = (
    experiment.groupby(["scenario", "model"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        sd_rmse=("rmse", "std"),
        mean_regret=("regret", "mean"),
        mean_expected_profit=("expected_profit", "mean"),
        oracle_profit=("oracle_profit", "mean"),
    )
    .sort_values(["scenario", "mean_regret", "mean_rmse"])
)
display(summary.round(4))

In [ ]:
for scenario in experiment["scenario"].unique():
    subset = summary[summary["scenario"] == scenario].sort_values("mean_regret")
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(subset["model"], subset["mean_regret"])
    ax.invert_yaxis()
    ax.set(
        xlabel="Mean expected regret per observation",
        title=f"Decision loss — {scenario.replace('_', ' ')}",
    )
    fig.tight_layout()
    plt.show()

selections = experiment.loc[
    experiment["model"] == "Decision-guided validation",
    ["scenario", "selected_candidate"],
]
display(
    selections.groupby(["scenario", "selected_candidate"])
    .size()
    .rename("times_selected")
    .reset_index()
)
assert (experiment["regret"] >= -1e-10).all()
assert np.isfinite(experiment[["rmse", "regret"]]).all().all()

## 6. What have we actually learned?

Interpret the tables using several distinct standards rather than one arbitrary aggregate importance score:

- **Structural:** which directions carry most X variance, and which are close to singular?
- **Inferential:** are coefficient intervals meaningful, and is a correlated group jointly informative?
- **Predictive:** which model generalises on held-out observations, with honest training-only tuning?
- **Stability:** do signs and estimates survive resampling?
- **Decision-theoretic:** does smaller forecast error translate into lower expected economic regret?

The causal question is separate. Correlation, coefficient significance and simulated prediction are insufficient to determine whether *intervening* changes an outcome.

### Limits and research programme

These cases intentionally use a linear conditional mean, Gaussian measurement noise, independent train/test draws, known costs and moderate `p/n`. More demanding experiments should vary predictor rank, signal strength, missingness mechanisms (MCAR/MAR/MNAR), outliers, heterogeneous interventions, covariate shift, nonlinear effects and `p/n`. For confirmatory claims, pre-register endpoints, use nested selection and evaluate independent datasets.

### References

Jolliffe (2002), *Principal Component Analysis*; Hoerl & Kennard (1970), ridge regression; Frank & Friedman (1993), chemometrics regression; Tibshirani (1996), Lasso; Zou & Hastie (2005), elastic net; Efron & Tibshirani (1993), bootstrap.

The full reference list and reproduction instructions are in the project README.